This code creates a set of xyz files with model polynuclear CexOy clusters from a bigger single clusters which are stored in 'CeO2.xyz' and 'Ce40.xyz' files for preparation of datasets A and B, respectively. The decision on how to slice an initial cluster is made based on the structure_catalogue instance created by structure_catalogue_maker() function from the [paper by Anker et al.](https://doi.org/10.1038/s41524-022-00896-3). As the atoms to be dropped from the initial big structure when constructing each smaller structure are selected randomly, some of the resulting structures may not be connected and consist of several atom 'islands'. In that case, the island containing the largest number of heavy atoms is retained with all coordinated oxygen atoms, while the smaller islands are dropped. The names of the prepared files are formatted so that they contain a number of Ce atoms in the cluster as the first symbol for easier labelling of data before the network training.

In [8]:
import numpy as np
import os
import random

# Import path configuration
import sys
sys.path.insert(0, '..')  # Add parent directory to path
from config import get_path

In [9]:
def format_XYZ(xyz_file, allowed_atoms):
    """
    Read an xyz file and count the number of atoms of specified types.
    
    Parameters
    ----------
    xyz_file : str
        Path to the xyz file
    allowed_atoms : list
        List of atom types to count (e.g., ["Ce"])
    
    Returns
    -------
    int
        Number of atoms matching the allowed types
    """
    atom_count = 0
    with open(xyz_file, 'r') as f:
        lines = f.readlines()
    
    for line in lines[2:]:  # Skip first two lines (atom count and comment)
        fields = line.split()
        if fields and fields[0] in allowed_atoms:
            atom_count += 1
    
    print(f"Found {atom_count} {allowed_atoms} atoms in {xyz_file}")
    return atom_count

In [10]:
def structure_catalogue_maker(Number_of_structures, Number_of_atoms, lower_atom_number, higher_atom_number):
    """This function makes a shuffled list containing 'Number_of_structures' number of lists which each is 
    'Number_of_atoms' long and is randomly distributed with 0's and 1's whereas the minimum number of 1's are 
    'lower_atom_number' and the maximum number of 1's are 'higher_atom_number'."""
    
    print(
        "Starting to make a structure catalogue with: ",
        f"{str(Number_of_structures)} structure from the starting model.",
    )
    print(
        f"The structure will have between {str(lower_atom_number)} and {str(higher_atom_number)} atoms"
    )
    structure_catalogue = []
    for _ in range(Number_of_structures):
        one_count = random.randint(lower_atom_number, higher_atom_number)
        zero_count = Number_of_atoms  - one_count
        my_list = [0]*zero_count + [1]*one_count
        random.shuffle(my_list)
        my_list.insert(0, one_count)
        structure_catalogue.append(my_list)
    print ("Permutations Succeeded")
    return structure_catalogue
    

In [11]:
def make_xyz_catalogue(catalogue, initial_xyz, output_dir, metal_atom, threshold_MO, threshold_MM):
    """
    Create xyz files for polynuclear clusters from a parent cluster structure.
    
    Parameters
    ----------
    catalogue : list
        List of structure masks from structure_catalogue_maker (each is a list starting with atom count, then 0/1 mask)
    initial_xyz : str
        Path to the initial large cluster xyz file (e.g., ce40.xyz)
    output_dir : Path
        Directory where output xyz files will be saved
    metal_atom : str
        Symbol of the metal atom (e.g., "Ce")
    threshold_MO : float
        Maximum metal-O distance to consider oxygen coordinated to metal
    threshold_MM : float
        Maximum metal-metal distance to consider atoms in same cluster
    """
    with open(initial_xyz, 'r') as f:
        lines = f.readlines()

    # Get the number of atoms from the xyz file
    num_atoms = int(lines[0])
    
    # Separate metal and non-metal atom lines
    metal_lines = []
    metal_indices = []
    for i, line in enumerate(lines[2:]):
        fields = line.split()
        if fields and fields[0] == metal_atom:
            metal_lines.append(line)
            metal_indices.append(i)
    
    created_count = 0

    # Iterate over the catalogue entries
    for k, entry in enumerate(catalogue):
        # entry[0] is the atom count, entry[1:] is the mask
        array = np.array(entry[1:])
        
        # Apply mask only to metal atoms
        selected_metal_lines = []
        for i, line in enumerate(metal_lines):
            if i < len(array) and array[i] == 1:
                selected_metal_lines.append(line)

        metal_coordinates = []
        for line in selected_metal_lines:
            fields = line.split()
            if fields:
                metal_coordinates.append([float(fields[1]), float(fields[2]), float(fields[3])])

        metal_groups = []
        while metal_coordinates:
            metal_group = [metal_coordinates.pop()]
            for metal_coordinate in metal_coordinates:
                distance = np.linalg.norm(np.array(metal_group[0]) - np.array(metal_coordinate))
                if distance < threshold_MM:
                    metal_group.append(metal_coordinate)
            metal_groups.append(metal_group)

        final_result = []
        for element in metal_groups:
            found = False
            for result in final_result:
                if set(map(tuple, element)) & set(map(tuple, result)):
                    result.extend([e for e in element if e not in result])
                    found = True
                    break
            if not found:
                final_result.append(element)

        if metal_groups:
            largest_metal_group = max(final_result, key=lambda x: len(x))
            
            # Count metal atoms for filename
            metal_count = len(largest_metal_group)
            
            # Save to output directory with nuclearity as first character
            output_file = output_dir / f'{metal_count}_{k}.xyz'
            with open(output_file, 'w') as new_f:
                # Write correct atom count
                total_atoms = len(largest_metal_group)
                # Count O atoms that will be included
                for line in lines[2:]:
                    fields = line.split()
                    if fields and fields[0] == "O":
                        O_coordinate = [float(fields[1]), float(fields[2]), float(fields[3])]
                        for metal_coordinate in largest_metal_group:
                            distance = np.linalg.norm(np.array(metal_coordinate) - np.array(O_coordinate))
                            if distance < threshold_MO:
                                total_atoms += 1
                                break
                
                new_f.write(f"{total_atoms}\n")
                new_f.write(lines[1])  # Comment line
                for metal_coordinate in largest_metal_group:
                    new_f.write(f"{metal_atom} " + " ".join([str(x) for x in metal_coordinate]) + "\n")
                for line in lines[2:]:
                    fields = line.split()
                    if fields and fields[0] == "O":
                        O_coordinate = [float(fields[1]), float(fields[2]), float(fields[3])]
                        for metal_coordinate in largest_metal_group:
                            distance = np.linalg.norm(np.array(metal_coordinate) - np.array(O_coordinate))
                            if distance < threshold_MO:
                                new_f.write(line)
                                break
            created_count += 1
    
    print(f"Created {created_count} xyz files in {output_dir}")

Generating clusters from the **CeO2** parent structure

In [13]:
# Set working directory to CeO2 clusters
ceo2_clusters_path = get_path('ceo2_clusters')
ceo2_output_path = get_path('ceo2_model_clusters')

# Ensure output directory exists
ceo2_output_path.mkdir(parents=True, exist_ok=True)

os.chdir(ceo2_clusters_path)
print(f"Working directory: {ceo2_clusters_path}")
print(f"Output directory: {ceo2_output_path}")
print(f"\nInput files expected:")
print(f"  - CeO2.xyz: {'exists' if (ceo2_clusters_path / 'ceo2.xyz').exists() else 'missing'}")

Working directory: /workspace/home/pdf-nn-data/ceo2_clusters
Output directory: /workspace/home/pdf-nn-data/ceo2_clusters/model_clusters

Input files expected:
  - CeO2.xyz: exists


In [14]:
# Parameters for CeO2 cluster generation
ceo2_starting_model = 'ceo2.xyz'
ceo2_allowed_atoms = ["Ce"]
Number_of_structures = 10000
print(f"Starting model: {ceo2_starting_model}")

Starting model: ceo2.xyz


In [17]:
# Generate structure catalogue for Ce clusters
NumCeO2 = format_XYZ(ceo2_starting_model, ceo2_allowed_atoms)
ceo2_structure_catalogue = structure_catalogue_maker(Number_of_structures, Number_of_atoms=NumCeO2, lower_atom_number=2, higher_atom_number=9)
# Generate Ce cluster xyz files from the catalogue
make_xyz_catalogue(
    catalogue=ceo2_structure_catalogue, 
    initial_xyz='ceo2.xyz',
    output_dir=ceo2_output_path,
    metal_atom="Ce",
    threshold_MO=3.0, 
    threshold_MM=5.2
)

Found 30 ['Ce'] atoms in ceo2.xyz
Starting to make a structure catalogue with:  10000 structure from the starting model.
The structure will have between 2 and 9 atoms
Permutations Succeeded
Created 10000 xyz files in /workspace/home/pdf-nn-data/ceo2_clusters/model_clusters


In [18]:
# Verify the created CeO2 cluster files
ceo2_directory = get_path('ceo2_model_clusters')
# Filter for files with expected naming pattern: number_number.xyz
ceo2_files = [f for f in os.listdir(ceo2_directory) if f.endswith('.xyz') and '_' in f and f.split('_')[0].isdigit()]
print(f"Total CeO2 xyz files created: {len(ceo2_files)}")

# Count by nuclearity
from collections import Counter
ceo2_nuclearities = [int(f.split('_')[0]) for f in ceo2_files]
ceo2_counts = Counter(ceo2_nuclearities)
print("\nDistribution by nuclearity:")
for nuc in sorted(ceo2_counts.keys()):
    print(f"  {nuc} Ce atoms: {ceo2_counts[nuc]} structures")

Total CeO2 xyz files created: 10000

Distribution by nuclearity:
  1 Ce atoms: 1860 structures
  2 Ce atoms: 2395 structures
  3 Ce atoms: 1861 structures
  4 Ce atoms: 1326 structures
  5 Ce atoms: 1006 structures
  6 Ce atoms: 737 structures
  7 Ce atoms: 440 structures
  8 Ce atoms: 260 structures
  9 Ce atoms: 115 structures


Generating clusters from the **Ce40** parent structure

In [19]:
# Set working directory to Ce clusters
ce_clusters_path = get_path('ce_clusters')
ce_output_path = get_path('ce_model_clusters')

# Ensure output directory exists
ce_output_path.mkdir(parents=True, exist_ok=True)

os.chdir(ce_clusters_path)
print(f"Working directory: {ce_clusters_path}")
print(f"Output directory: {ce_output_path}")
print(f"\nInput files expected:")
print(f"  - ce40.xyz: {'exists' if (ce_clusters_path / 'ce40.xyz').exists() else 'missing'}")

Working directory: /workspace/home/pdf-nn-data/ce_clusters
Output directory: /workspace/home/pdf-nn-data/ce_clusters/model_clusters

Input files expected:
  - ce40.xyz: exists


In [20]:
# Parameters for Ce cluster generation
ce_starting_model = 'ce40.xyz'
ce_allowed_atoms = ["Ce"]
print(f"Starting model: {ce_starting_model}")

Starting model: ce40.xyz


In [21]:
# Generate structure catalogue for Ce clusters
NumCe = format_XYZ(ce_starting_model, ce_allowed_atoms)
ce_structure_catalogue = structure_catalogue_maker(Number_of_structures, Number_of_atoms=NumCe, lower_atom_number=2, higher_atom_number=9)

Found 40 ['Ce'] atoms in ce40.xyz
Starting to make a structure catalogue with:  10000 structure from the starting model.
The structure will have between 2 and 9 atoms
Permutations Succeeded


In [22]:
# Generate Ce cluster xyz files from the catalogue
make_xyz_catalogue(
    catalogue=ce_structure_catalogue, 
    initial_xyz='ce40.xyz',
    output_dir=ce_output_path,
    metal_atom="Ce",
    threshold_MO=3.0, 
    threshold_MM=5.2
)

Created 10000 xyz files in /workspace/home/pdf-nn-data/ce_clusters/model_clusters


In [23]:
# Verify the created Ce cluster files
ce_directory = get_path('ce_model_clusters')
ce_files = [f for f in os.listdir(ce_directory) if f.endswith('.xyz')]
print(f"Total Ce xyz files created: {len(ce_files)}")

# Count by nuclearity
from collections import Counter
ce_nuclearities = [int(f.split('_')[0]) for f in ce_files]
ce_counts = Counter(ce_nuclearities)
print("\nDistribution by nuclearity:")
for nuc in sorted(ce_counts.keys()):
    print(f"  {nuc} Ce atoms: {ce_counts[nuc]} structures")

Total Ce xyz files created: 10000

Distribution by nuclearity:
  1 Ce atoms: 2188 structures
  2 Ce atoms: 2614 structures
  3 Ce atoms: 2100 structures
  4 Ce atoms: 1433 structures
  5 Ce atoms: 857 structures
  6 Ce atoms: 472 structures
  7 Ce atoms: 215 structures
  8 Ce atoms: 95 structures
  9 Ce atoms: 26 structures
